# ER independent evidence review — two source folders

Execution environment: Google Colab. The notebook contains a copy of the human annotation workbook, searches original PRD/meeting and Jira source folders separately, and exports an evidence sheet for `UNSURE` pairs. It **never reads resolver predictions** or automatically assigns SAME/DIFFERENT. Evidence retrieval is not human verification.

In [1]:
from pathlib import Path
import base64, json, re, csv
from collections import defaultdict
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.worksheet.datavalidation import DataValidation
try:
    from google.colab import drive, files
    drive.mount('/content/drive')
except ImportError:
    files = None
BASE = Path('/content/drive/MyDrive/GC1')
PRD_MEETING_ROOT = BASE / 'GC1_Generated_PRD_Meetings'
JIRA_ROOT = BASE / 'GC1_Jira_Dataset'
for folder in [PRD_MEETING_ROOT, JIRA_ROOT]:
    assert folder.is_dir(), f'Missing folder: {folder}'
    print('OK:', folder)
OUTPUT_FILE = Path('/content/ER_independent_source_evidence_review.xlsx')


Mounted at /content/drive
OK: /content/drive/MyDrive/GC1/GC1_Generated_PRD_Meetings
OK: /content/drive/MyDrive/GC1/GC1_Jira_Dataset


In [4]:
ANNOTATION_FILE = Path('/content/ER_human_annotations_original.xlsx')
ANNOTATION_FILE.write_bytes(base64.b64decode(ANNOTATION_BYTES_B64))
wb = openpyxl.load_workbook(ANNOTATION_FILE, data_only=False)
ws = wb['Human annotation']
headers = [str(c.value).strip() for c in ws[1]]
rows = [dict(zip(headers, row)) for row in list(ws.values)[1:]]
unsure = [r for r in rows if str(r.get('human_label') or '').strip().upper() == 'UNSURE']
print('All pairs:', len(rows), '| UNSURE:', len(unsure))
assert unsure, 'No UNSURE rows in the embedded workbook'

All pairs: 144 | UNSURE: 50


## Find original source files
Only the six selected projects are searched. Prompt files, metadata, extraction outputs, and resolver files are excluded. Jira source formats differ between projects; unsupported formats will be flagged for manual lookup.

In [5]:
PROJECT_FOLDERS = {'JBIDE':'02_JBIDE','SAK':'03_SAK','MNG':'05_MNG','MC':'10_MC','HBASE':'18_HBASE','SERVER':'20_SERVER'}
SKIP = ('.prompt.', '.meta.', 'extracted', 'resolved', 'canonical', 'prediction', 'annotation', 'evaluation')
source_index = defaultdict(list)
for project, dirname in PROJECT_FOLDERS.items():
    folder = PRD_MEETING_ROOT / dirname
    if not folder.exists():
        print('Missing PRD/meeting project:', folder)
        continue
    for path in folder.rglob('*'):
        if path.is_file() and path.suffix.lower() in ('.txt', '.md') and not any(x in path.name.lower() for x in SKIP):
            source_index[(project, path.stem.lower())].append(path)
print('Original PRD/meeting source IDs indexed:', len(source_index))

def norm(s):
    return re.sub(r'\s+', ' ', str(s or '')).strip()

def excerpts(text, name, window=300, limit=3):
    name = norm(name)
    if not name: return []
    pattern = re.compile(r'(?<![A-Za-z0-9])' + re.escape(name) + r'(?![A-Za-z0-9])', re.I)
    result=[]
    for m in pattern.finditer(text):
        result.append(norm(text[max(0,m.start()-window):min(len(text),m.end()+window)]))
        if len(result) >= limit: break
    return result

def read_text(path):
    return path.read_text(encoding='utf-8', errors='replace')

# Index individual Jira issue files by filename, without reading whole Jira dataset.
jira_file_index = defaultdict(list)
for project, dirname in PROJECT_FOLDERS.items():
    folder = JIRA_ROOT / dirname
    if not folder.exists():
        print('Missing Jira project:', folder)
        continue
    for path in folder.rglob('*'):
        if path.is_file() and path.suffix.lower() in ('.json','.jsonl','.txt','.md') and not any(x in path.name.lower() for x in SKIP):
            jira_file_index[(project, path.stem.upper())].append(path)
print('Jira filename stems indexed:', len(jira_file_index))


Original PRD/meeting source IDs indexed: 18
Jira filename stems indexed: 24


In [6]:
def get_jira_record(project, issue_key):
    # First try original per-issue files.
    candidates = jira_file_index.get((project, issue_key.upper()), [])
    for path in candidates:
        if path.stat().st_size > 30_000_000: continue
        return read_text(path), str(path)
    # Then look for original records in JSON / JSONL collections for this project.
    # A strict key check avoids returning the wrong issue merely because it was mentioned.
    folder = JIRA_ROOT / PROJECT_FOLDERS[project]
    for path in folder.rglob('*'):
        if not path.is_file() or path.suffix.lower() not in ('.json','.jsonl'): continue
        if any(x in path.name.lower() for x in SKIP) or path.stat().st_size > 50_000_000: continue
        try:
            if path.suffix.lower() == '.jsonl':
                with path.open(encoding='utf-8',errors='replace') as fh:
                    for line in fh:
                        if issue_key.upper() not in line.upper(): continue
                        try: obj=json.loads(line)
                        except json.JSONDecodeError: continue
                        if isinstance(obj,dict) and str(obj.get('key') or obj.get('issue_key') or '').upper()==issue_key.upper():
                            return json.dumps(obj,ensure_ascii=False),str(path)
            else:
                obj=json.loads(read_text(path))
                if isinstance(obj,dict):
                    arr=obj.get('issues', obj.get('records', []))
                    if str(obj.get('key') or obj.get('issue_key') or '').upper()==issue_key.upper():
                        return json.dumps(obj,ensure_ascii=False),str(path)
                else: arr=obj
                if isinstance(arr,list):
                    for item in arr:
                        if isinstance(item,dict) and str(item.get('key') or item.get('issue_key') or '').upper()==issue_key.upper():
                            return json.dumps(item,ensure_ascii=False),str(path)
        except (OSError,ValueError,TypeError): pass
    return None, ''

def find_evidence(project, source_type, source_id, name):
    project = str(project).upper()
    st = str(source_type or '').upper()
    sid = str(source_id or '').removesuffix('.txt').strip()
    if st == 'JIRA':
        text, path = get_jira_record(project, sid)
        if text is None: return 'SOURCE_NOT_FOUND','',''
    else:
        paths = source_index.get((project, sid.lower()), [])
        if not paths: return 'SOURCE_NOT_FOUND','',''
        path = str(paths[0]); text = read_text(paths[0])
    hits = excerpts(text, name)
    return ('FOUND' if hits else 'TERM_NOT_FOUND'),path,'\n---\n'.join(hits)


## Export the evidence workbook
The final annotation column is blank in the supplied review template; supported labels are SAME, DIFFERENT and UNSURE, accompanied by a brief justification. If a source is missing, do not guess.

In [7]:
out=openpyxl.Workbook(); sheet=out.active;sheet.title='UNSURE evidence review'
columns=['pair_id','project','entity_type','name_a','name_b','source_type_a','source_id_a','source_path_a','evidence_status_a','excerpt_a','source_type_b','source_id_b','source_path_b','evidence_status_b','excerpt_b','final_human_label','human_evidence_reason','reviewer']
sheet.append(columns)
counts=defaultdict(int)
for r in unsure:
    a,ap,ae=find_evidence(r['project'],r['source_a'],r['source_id_a'],r['name_a'])
    b,bp,be=find_evidence(r['project'],r['source_b'],r['source_id_b'],r['name_b'])
    counts[a]+=1;counts[b]+=1
    sheet.append([r['pair_id'],r['project'],r['entity_type'],r['name_a'],r['name_b'],r['source_a'],r['source_id_a'],ap,a,ae,r['source_b'],r['source_id_b'],bp,b,be,'','',''])
sheet.freeze_panes='D2';sheet.auto_filter.ref=sheet.dimensions
for c in sheet[1]:
    c.fill=PatternFill('solid',fgColor='17324F');c.font=Font(color='FFFFFF',bold=True);c.alignment=Alignment(wrap_text=True)
widths={'A':13,'B':12,'C':18,'D':28,'E':28,'F':16,'G':25,'H':52,'I':20,'J':85,'K':16,'L':25,'M':52,'N':20,'O':85,'P':20,'Q':58,'R':20}
for col,width in widths.items(): sheet.column_dimensions[col].width=width
for row in sheet.iter_rows(min_row=2):
    for cell in row: cell.alignment=Alignment(vertical='top',wrap_text=True)
    sheet.row_dimensions[row[0].row].height=80
dv=DataValidation(type='list',formula1='"SAME,DIFFERENT,UNSURE"');sheet.add_data_validation(dv);dv.add(f'P2:P{sheet.max_row}')
notes=out.create_sheet('Instructions')
for item in [('Purpose','Independent original-source evidence for unresolved ER pairs'),('Do not use','Resolver predictions, canonical IDs, embedding scores, sampling group'),('Human decision','Fill final_human_label and human_evidence_reason based on source evidence'),('SOURCE_NOT_FOUND','Original source not found in current folders; do not infer DIFFERENT'),('TERM_NOT_FOUND','Document found but exact name absent; inspect aliases manually'),('SAME','Evidence establishes same real-world entity'),('DIFFERENT','Evidence establishes distinct entities'),('UNSURE','Evidence is insufficient to decide')]: notes.append(item)
notes.column_dimensions['A'].width=25;notes.column_dimensions['B'].width=90
out.save(OUTPUT_FILE)
print('Saved:',OUTPUT_FILE,'| Pairs:',len(unsure),'| Endpoint evidence statuses:',dict(counts))
if files is not None: files.download(str(OUTPUT_FILE))


Saved: /content/ER_independent_source_evidence_review.xlsx | Pairs: 50 | Endpoint evidence statuses: {'FOUND': 100}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>